In [ ]:
from typing import Union
import numpy as np

from scipy.sparse import spmatrix

# An NDArray can either be a numpy array (np.ndarray) or a sparse matrix (spmatrix)
NDArray = Union[np.ndarray, spmatrix]

np.random.seed(42)


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion

import pandas as pd
import os


In [ ]:
TRAIN_DATASET_SOURCE = os.path.join("data/ling-539-competition-2026", "train.csv")
TEST_DATASET_SOURCE = os.path.join("data/ling-539-competition-2026", "test.csv")
RANDOM_STATE = 42

CLASS2NAME = {0: "not a review", 1: "positive", 2:"negative"}
NAME2CLASS = {name: label for label, name in CLASS2NAME.items()}

pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)

train_data = pd.read_csv(TRAIN_DATASET_SOURCE, keep_default_na=False, encoding='utf-8')
test_data = pd.read_csv(TEST_DATASET_SOURCE, keep_default_na=False, encoding='utf-8')

train_texts = train_data['TEXT']
train_labels = train_data['LABEL']
test_texts = test_data['TEXT']
test_ids = test_data['ID']

# Quick Inspection to see how the data is loaded in...
print("Training Data Info:")

train_data.info()
print(f"Train axes: {train_data.axes}")
print(f"Train columns: {train_data.columns}")

print("Test Data Info: ")

test_data.info()
print(f"Test axes: {test_data.axes}")
print(f"Test columns: {test_data.columns}")


In [ ]:
class Classifier:
    def __init__(self):
        """
        Initializes a classifier object..
        """

        self.word_vec = TfidfVectorizer(input='content', ngram_range=(1,2),  min_df=3, max_df=0.7, sublinear_tf=True)
        self.char_vec = TfidfVectorizer(input='content', ngram_range=(3,5), min_df=10, analyzer='char', sublinear_tf=True, max_features=40000)
        self.vectorizer = FeatureUnion([("words_vec", self.word_vec), ("chars_vec", self.char_vec)])
        
        self.le = LabelEncoder()
        
        self.model = LogisticRegression(C=2, l1_ratio=0, solver='sag', class_weight='balanced', random_state=RANDOM_STATE, max_iter=1000)

    def train(self, training_texts, training_labels):
        """
        Converts training texts and labels to vectors.
        Fits classifier model with these vectors.
        Returns vectorized text features and labels.
        """
        X = self.vectorizer.fit_transform(training_texts)
        y = self.le.fit_transform(training_labels)
        self.model.fit(X, y)
        return (X,y)

    def predict(self, input_texts):
        """
        Predicts labels for each input text. 
        :param tokens: A sequence of text documents. Each document is a review. 
        :return: A sequence of predicted labels, one for each review/document.
        """
        X = self.vectorizer.transform(input_texts)
        labels = self.model.predict(X)
        return labels


In [ ]:
# fit/"train" the feature extractor and label encoder with training data
clf = Classifier()
feats, labels = clf.train(train_texts, train_labels)

# predict labels for test data
predicted_indices = clf.predict(test_texts)


In [ ]:
# Store prediction in Dataframe and write Dataframe to file
test_out = pd.DataFrame(
    {'ID': test_ids, 'LABEL': predicted_indices}
)
test_out.to_csv("predictions.csv", index=False)